# Phase 1: Data Cleaning and Campus Selection

This notebook inspects site completeness across all BDG2 sites, selects the campus site (`Fox`), and verifies the processed dataset.

In [ ]:
import sys
from pathlib import Path
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root))

import pandas as pd
from src.data_loader import site_completeness, select_campus
from src import config

# 1. Load site completeness and sort by number of buildings with electricity meters
completeness_df = site_completeness().sort_values('n_buildings_elec', ascending=False)
completeness_df

**Observation:** In electricity_cleaned.csv only 5 of 19 sites have mean completeness of at least 0.95 (in the raw file 12 of 19 do) (Wolf 0.996, Robin 0.989, Fox 0.983, Bull 0.965, Bear 0.958). The weakest are Swan 0.478, Shrew 0.544, Bobcat 0.716, Mouse 0.754 and Panther 0.802.

In [ ]:
# 2. Plot mean completeness and campus usage share per site
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(12, 6))
ax1.bar(completeness_df.index, completeness_df['mean_completeness'], color='b', alpha=0.6, label='Mean Completeness')
ax1.set_ylabel('Mean Completeness', color='b')
plt.xticks(rotation=45, ha='right')

ax2 = ax1.twinx()
ax2.plot(completeness_df.index, completeness_df['campus_usage_share'], color='r', marker='o', label='Campus Usage Share')
ax2.set_ylabel('Education/Office Share', color='r')

plt.title('Site Completeness and Campus Usage Share')
plt.tight_layout()
plt.show()

**Observation:** Fox combines high completeness (0.983) with a 63.5% Education/Office share and 137 electricity buildings. Wolf and Robin score higher on completeness and campus share but have only 36 and 52 buildings.

## Justification for Site Selection: Fox

- **Number of electricity buildings:** 137
- **Mean completeness:** 98.3%
- **High completeness share:** 95.6% of buildings have >= 90% non-missing hours
- **Campus usage share:** 63.5% Education or Office
- **Weather coverage:** 17,543 weather rows

**Limitations:** 
- Located in the US/Mountain climate zone, which may not generalize to other climates.
- 63.5% of Fox's buildings are Education or Office, but the metadata has no field confirming a university, so Fox is campus-like, not confirmed as a university.

In [ ]:
# 4. Select campus 'Fox' and review selection report
kept_meta, report = select_campus('Fox', min_completeness=0.9)
print("Selection Report:")
for k, v in report.items():
    print(f"{k}: {v}")

**Observation:** Out of 137 buildings at Fox, 131 were kept (>=90% completeness) and 6 were dropped due to low completeness. Zero buildings were dropped due to missing meters.

In [ ]:
# 5. Load processed datasets and print summary statistics
long_df = pd.read_parquet(config.CAMPUS_LONG_PARQUET)
total_df = pd.read_parquet(config.CAMPUS_TOTAL_PARQUET)

print("--- Campus Long Parquet ---")
print(f"Shape: {long_df.shape}")
print(long_df.dtypes)
print(f"NaN share: {long_df['kwh'].isnull().mean():.4f}")
print(f"Date range: {long_df['timestamp'].min()} to {long_df['timestamp'].max()}")

print("\n--- Campus Total Parquet ---")
print(f"Shape: {total_df.shape}")
print(total_df.dtypes)
print(f"NaN share: {total_df['kwh'].isnull().mean():.4f}")
print(f"Date range: {total_df['timestamp'].min()} to {total_df['timestamp'].max()}")

**Observation:** The long table has 2,298,264 rows (131 buildings x 17,544 hours). kwh is about 0.4% NaN and the campus total has 7 NaN hours.